In [1]:
import polars as pl
from pathlib import Path

DATA_PATH = "M1_M4.csv"
OUTPUT_DIR = Path("data_prep")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────
df = pl.read_csv(DATA_PATH)

print(f"Program participants: {len(df)}")



# ── Target ────────────────────────────────────────────────────────────────
df = df.with_columns(
    pl.when(pl.col("module") == "M1")
      .then(pl.col("stats_m1_module_completed_flag").cast(pl.Int8))
      .when(pl.col("module") == "M2")
      .then(pl.col("stats_m2_module_completed_flag").cast(pl.Int8))
      .otherwise(pl.lit(None).cast(pl.Int8))
      .alias("target")
)

# ── Hist features (build BEFORE removing soft leakage) ───────────────────
HIST_M1_MAP = {
    "stats_m1_row_count":                 "hist_m1_row_count",
    "stats_m1_lessons_viewed_count":      "hist_m1_lessons_viewed_count",
    "stats_m1_content_viewed_units":      "hist_m1_content_viewed_units",
    "stats_m1_final_tasks_solved_count":  "hist_m1_final_tasks_solved_count",
    "stats_m1_interim_assessment_score":  "hist_m1_interim_assessment_score",
}
HIST_M2_MAP = {
    "stats_m2_row_count":                 "hist_m2_row_count",
    "stats_m2_lessons_viewed_count":      "hist_m2_lessons_viewed_count",
    "stats_m2_content_viewed_units":      "hist_m2_content_viewed_units",
    "stats_m2_final_tasks_solved_count":  "hist_m2_final_tasks_solved_count",
    "stats_m2_interim_assessment_score":  "hist_m2_interim_assessment_score",
}

# Keep only existing columns
HIST_M1_MAP = {k: v for k, v in HIST_M1_MAP.items() if k in df.columns}
HIST_M2_MAP = {k: v for k, v in HIST_M2_MAP.items() if k in df.columns}

m1_hist = (
    df.filter(pl.col("module") == "M1")
    .select(["user_id"] + list(HIST_M1_MAP.keys()))
    .rename(HIST_M1_MAP)
)
m2_hist = (
    df.filter(pl.col("module") == "M2")
    .select(["user_id"] + list(HIST_M2_MAP.keys()))
    .rename(HIST_M2_MAP)
)

df = df.join(m1_hist, on="user_id", how="left")
df = df.join(m2_hist, on="user_id", how="left")

# Null out hist_m1_* for M1 rows (they have no prior M1 history)
hist_m1_cols = [c for c in df.columns if c.startswith("hist_m1_")]
hist_m2_cols = [c for c in df.columns if c.startswith("hist_m2_")]

df = df.with_columns([
    pl.when(pl.col("module") == "M1")
      .then(pl.lit(None).cast(df[c].dtype))
      .otherwise(pl.col(c))
      .alias(c)
    for c in hist_m1_cols
])

df = df.with_columns([
    pl.when(pl.col("module").is_in(["M1", "M2"]))
      .then(pl.lit(None).cast(df[c].dtype))
      .otherwise(pl.col(c))
      .alias(c)
    for c in hist_m2_cols
])

# ── Hard & soft leakage removal ───────────────────────────────────────────
HARD_LEAKAGE_BASE = [
    "wk_course_completed_at",
    "uc_has_completion_record",
    "uc_completion_delay_days",
    "uc_completed_flag",
    "uc_completed_without_points_flag",
    "uc_days_from_anchor_to_completion",
]

HARD_LEAKAGE_TEMPLATES = [
    "{}_module_status",
    "{}_module_completed_flag",
    "{}_module_dropout_flag",
    "{}_viewed_80pct_lessons_or_video_flag",
    "{}_viewed_80pct_lessons_or_video_negative_flag",
    "{}_attended_live_lesson_flag",
    "{}_attended_live_lesson_negative_flag",
    "{}_all_required_final_tasks_solved_flag",
    "{}_all_required_final_tasks_solved_negative_flag",
    "{}_current_control_passed_flag",
    "{}_current_control_passed_negative_flag",
    "{}_interim_assessment_passed_flag",
    "{}_interim_assessment_passed_negative_flag",
]

SOFT_LEAKAGE_TEMPLATES = [
    "{}_row_count",
    "{}_lessons_viewed_count",
    "{}_content_viewed_units",
    "{}_final_tasks_solved_count",
    "{}_interim_assessment_score",
]

cols_to_drop = list(HARD_LEAKAGE_BASE)
for prefix in ["stats_m1", "stats_m2", "stats_m3"]:
    for tmpl in HARD_LEAKAGE_TEMPLATES + SOFT_LEAKAGE_TEMPLATES:
        cols_to_drop.append(tmpl.format(prefix))

cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(cols_to_drop)
print(f"Dropped {len(cols_to_drop)} leakage columns. Remaining: {df.shape[1]}")

# ── Numeric module feature ────────────────────────────────────────────────
df = df.with_columns(
    pl.when(pl.col("module") == "M1").then(pl.lit(1))
      .when(pl.col("module") == "M2").then(pl.lit(2))
      .when(pl.col("module") == "M3").then(pl.lit(3))
      .otherwise(pl.lit(0))
      .cast(pl.Int8)
      .alias("module_num")
)

# ── Split ─────────────────────────────────────────────────────────────────
df_train = df.filter(pl.col("module").is_in(["M1", "M2"]))
df_infer = df.filter(pl.col("module") == "M3")

# ── Sanity checks ─────────────────────────────────────────────────────────
m1_users = set(df.filter(pl.col("module") == "M1")["user_id"].to_list())
m2_users = set(df.filter(pl.col("module") == "M2")["user_id"].to_list())
m3_users = set(df.filter(pl.col("module") == "M3")["user_id"].to_list())

print(f"\n── User intersection ──")
print(f"  M1: {len(m1_users)}, M2: {len(m2_users)}, M3: {len(m3_users)}")
print(f"  M2 ∩ M1 : {len(m2_users & m1_users)}  (expected = all M2)")
print(f"  M3 ∩ M2 : {len(m3_users & m2_users)}  (expected = all M3)")
print(f"  M3 ∩ M1 : {len(m3_users & m1_users)}")

print(f"\n── Class balance (df_train) ──")
balance = (
    df_train.filter(pl.col("target").is_not_null())
    .group_by(["module", "target"])
    .agg(pl.len().alias("n"))
    .sort(["module", "target"])
    .with_columns(
        (pl.col("n") / pl.col("n").sum().over("module") * 100)
        .round(1)
        .alias("pct")
    )
)
print(balance)

print(f"\n── Hist features null check ──")
for c in hist_m1_cols + hist_m2_cols:
    if c not in df_train.columns:
        continue
    nulls_by_module = (
        df_train.group_by("module")
        .agg(pl.col(c).null_count().alias("null_n"), pl.len().alias("total"))
        .sort("module")
    )
    print(f"  {c}:  {nulls_by_module.to_dict(as_series=False)}")

# ── Save ──────────────────────────────────────────────────────────────────
df_train.write_parquet(OUTPUT_DIR / "df_train.parquet")
df_infer.write_parquet(OUTPUT_DIR / "df_infer.parquet")
print(f"\nSaved: df_train {df_train.shape}, df_infer {df_infer.shape} → {OUTPUT_DIR}/")

Program participants: 6712
Dropped 51 leakage columns. Remaining: 452

── User intersection ──
  M1: 2972, M2: 1955, M3: 1785
  M2 ∩ M1 : 1955  (expected = all M2)
  M3 ∩ M2 : 1785  (expected = all M3)
  M3 ∩ M1 : 1785

── Class balance (df_train) ──
shape: (4, 4)
┌────────┬────────┬──────┬──────┐
│ module ┆ target ┆ n    ┆ pct  │
│ ---    ┆ ---    ┆ ---  ┆ ---  │
│ str    ┆ i8     ┆ u32  ┆ f64  │
╞════════╪════════╪══════╪══════╡
│ M1     ┆ 0      ┆ 1017 ┆ 34.2 │
│ M1     ┆ 1      ┆ 1955 ┆ 65.8 │
│ M2     ┆ 0      ┆ 170  ┆ 8.7  │
│ M2     ┆ 1      ┆ 1785 ┆ 91.3 │
└────────┴────────┴──────┴──────┘

── Hist features null check ──
  hist_m1_row_count:  {'module': ['M1', 'M2'], 'null_n': [2972, 0], 'total': [2972, 1955]}
  hist_m1_lessons_viewed_count:  {'module': ['M1', 'M2'], 'null_n': [2972, 0], 'total': [2972, 1955]}
  hist_m1_content_viewed_units:  {'module': ['M1', 'M2'], 'null_n': [2972, 0], 'total': [2972, 1955]}
  hist_m1_final_tasks_solved_count:  {'module': ['M1', 'M2'], 'null_